# YOLOv3 多尺度

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/10-multiscale/)

先執行下一格的環境格，再閱讀、執行最後的完整實驗。各節互相獨立，不需要先跑其他節。

最後一格存著這段程式在 CPU 上實際執行的輸出；執行環境與其他實驗紀錄見網站的〈驗證範圍〉。範例不用預訓練權重，也不需要下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.5.0'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 各節實驗都在 CPU 上執行，執行紀錄用的是 PyTorch 2.9.1。已經是 2.9.1 就沿用（CPU 或 CUDA 版都可以）；
# 其他版本會改裝成 2.9.1 的 CPU 版，之後這個工作階段就用不到 GPU，但各節實驗本來就不需要。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

### 可選：兩尺度 40 步與實際預測

跑完完整實驗後，另開一個 code cell：

```python
!python scripts/run_multiscale_learning.py
from IPython.display import SVG, display
display(SVG(filename='artifacts/runs/multiscale-learning/learning.svg'))
```

這只訓練一張圖；小框不一定達到 IoU 0.5，請照實際的報告解讀。

In [1]:
import torch
from torch import nn
from miniyolo.targets import build_targets
from miniyolo.losses import grid_loss
from miniyolo.inference import decode_grid
from miniyolo.geometry import nms


class TwoScale(nn.Module):
    def __init__(self):
        super().__init__()
        self.early = nn.Sequential(nn.Conv2d(3,8,3,2,1),nn.ReLU(),
                                   nn.Conv2d(8,16,3,2,1),nn.ReLU(),
                                   nn.Conv2d(16,16,3,2,1),nn.ReLU())
        self.deep = nn.Sequential(nn.Conv2d(16,32,3,2,1),nn.ReLU())
        self.fine_head = nn.Conv2d(16,7,1)
        self.coarse_head = nn.Conv2d(32,7,1)

    def forward(self,x):
        fine = self.early(x)
        coarse = self.deep(fine)
        return (self.fine_head(fine).permute(0,2,3,1),
                self.coarse_head(coarse).permute(0,2,3,1))


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    small = {'boxes':torch.tensor([[5.,5.,13.,13.]]),'labels':torch.tensor([0])}
    large = {'boxes':torch.tensor([[32.,32.,56.,56.]]),'labels':torch.tensor([1])}
    coarse_small = build_targets([small],4,64,2)
    fine_small = build_targets([small],8,64,2)
    assert torch.allclose(coarse_small['box'][0,0,0],torch.tensor([.5625,.5625,.125,.125]))
    assert torch.allclose(fine_small['box'][0,1,1],torch.tensor([.125,.125,.125,.125]))
    image = torch.zeros(1,3,64,64)
    image[0,0,5:13,5:13] = 1
    image[0,2,32:56,32:56] = 1
    model = TwoScale()
    optimizer = torch.optim.Adam(model.parameters(),lr=.01)
    fine,coarse = model(image)
    assert fine.shape == (1,8,8,7) and coarse.shape == (1,4,4,7)
    # Disjoint scale assignment: small box to fine, large to coarse.
    coarse_large = build_targets([large],4,64,2)
    assert fine_small['objectness'][0,5,5] == 0  # large GT is unassigned to fine
    assert coarse_large['objectness'][0,0,0] == 0  # small GT is unassigned to coarse
    swapped_small = {'boxes':small['boxes'],'labels':torch.tensor([1])}
    assert torch.equal(build_targets([swapped_small],8,64,2)['positive'],fine_small['positive'])
    loss = grid_loss(fine,fine_small)['total'] + grid_loss(coarse,coarse_large)['total']
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    assert model.fine_head.weight.grad.abs().sum() > 0
    assert model.coarse_head.weight.grad.abs().sum() > 0
    optimizer.step()
    # Encode the SAME small GT to each scale and check both decode paths.
    decoded_scales = []
    for size,target in [(4,coarse_small),(8,fine_small)]:
        raw = torch.zeros(1,size,size,7)
        raw[...,4] = -20  # channel 4 is objectness logit
        b,y,x = target['positive'].nonzero()[0].tolist()
        raw[b,y,x,:4] = torch.logit(target['box'][b,y,x])
        raw[b,y,x,4:] = torch.tensor([10.,10.,-10.])  # obj, class0, class1
        decoded = decode_grid(raw,64,.25,.5)[0]
        assert torch.allclose(decoded['boxes'],small['boxes'],atol=1e-4)
        decoded_scales.append(decoded)
    boxes = torch.cat([p['boxes'] for p in decoded_scales],dim=0)
    scores = torch.cat([p['scores'] for p in decoded_scales],dim=0)
    labels = torch.cat([p['labels'] for p in decoded_scales],dim=0)
    assert boxes.shape == (2,4) and scores.shape == labels.shape == (2,)
    selected = []
    for label in labels.unique():
        indices = torch.where(labels==label)[0]
        selected.append(indices[nms(boxes[indices],scores[indices],iou_threshold=.5)])
    selected = torch.cat(selected)
    assert len(selected) == 1
    assert torch.allclose(boxes[selected],small['boxes'],atol=1e-4)
    candidates = fine.shape[1]*fine.shape[2] + coarse.shape[1]*coarse.shape[2]
    assert candidates == 80
    print('cross-scale duplicate boxes before/after class-wise NMS',len(boxes),'->',len(selected))
    print('stride16 small target',coarse_small['box'][0,0,0].tolist())
    print('stride8 small target',fine_small['box'][0,1,1].tolist())
    print('head shapes',tuple(fine.shape),tuple(coarse.shape),'candidates before score/NMS',candidates)
    print('both heads received gradients; one update; scale encode/decode checked')


if __name__ == '__main__':
    main()


cross-scale duplicate boxes before/after class-wise NMS 2 -> 1
stride16 small target [0.5625, 0.5625, 0.125, 0.125]
stride8 small target [0.125, 0.125, 0.125, 0.125]
head shapes (1, 8, 8, 7) (1, 4, 4, 7) candidates before score/NMS 80
both heads received gradients; one update; scale encode/decode checked
